# CornerScout · 02 Limpieza, contexto preevento y EDA estructural

**Responsabilidad única:** normalizar los eventos de 01, auditar estructura y
calidad, y reconstruir el marcador y los jugadores en campo inmediatamente
antes de cada evento. SCR-15 no se calcula aquí.

Las salidas normalizadas son el contrato exclusivo de entrada del notebook 03.

## 0 · Entorno y rutas

No se requieren descargas ni credenciales. Se lee un partido por vez para no
cargar 1.3 millones de eventos anchos en memoria.

In [ ]:
import importlib.util
import subprocess
import sys

PACKAGES = {
    "numpy": "numpy>=1.26,<3",
    "pandas": "pandas>=2.2,<4",
    "matplotlib": "matplotlib>=3.8,<4",
    "pyarrow": "pyarrow>=16",
    "httpx": "httpx>=0.28,<1",
    "tqdm": "tqdm>=4.66",
}
missing = [spec for module, spec in PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencias:", "instaladas las ausentes" if missing else "ya disponibles")

In [ ]:
import gzip
import hashlib
import json
import os
import platform
from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_DIR = Path(os.environ.get(
    "CORNERSCOUT_DATA_DIR",
    "/content/drive/MyDrive/Corner_Scout/data" if IN_COLAB else "data",
))
RAW_DIR = DATA_DIR / "raw"
INGESTION_DIR = DATA_DIR / "interim" / "01_ingestion"
OUT = DATA_DIR / "interim" / "02_clean"
EVENTS_OUT = OUT / "events"
FIG_DIR = OUT / "figures"
for path in (OUT, EVENTS_OUT, FIG_DIR):
    path.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
CONTEXT_VERSION = "match-context-v1-on-pitch"
ENVIRONMENT = {name: version(name) for name in
               ["numpy", "pandas", "matplotlib", "pyarrow", "tqdm"]}
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})
display(pd.Series({"raw": str(RAW_DIR), "salida": str(OUT), "run_id": RUN_ID,
                   "contexto": CONTEXT_VERSION, "python": platform.python_version(),
                   **ENVIRONMENT}))

## 1 · Contrato de entrada

Se verifica el contrato de 01 y la identidad de los 380 archivos. La fecha y
hora del proveedor determinan el orden de partidos; dentro de cada partido
manda `period, index`.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


ingestion_contract_path = INGESTION_DIR / "contract.json"
assert ingestion_contract_path.is_file(), "Ejecuta primero 01_ingesta_statsbomb.ipynb"
ingestion_contract = json.loads(ingestion_contract_path.read_text(encoding="utf-8"))
assert ingestion_contract["counts"]["matches"] == 380
assert ingestion_contract["counts"]["event_files"] == 380

matches_raw = pd.read_csv(RAW_DIR / "matches_laliga_2015_16.csv")
for side in ("home", "away"):
    alternative = f"{side}_team_{side}_team_name"
    if f"{side}_team" not in matches_raw and alternative in matches_raw:
        matches_raw[f"{side}_team"] = matches_raw[alternative]
required = ["match_id", "match_date", "kick_off", "home_team", "away_team",
            "home_score", "away_score"]
assert set(required) <= set(matches_raw)
matches = matches_raw[required].copy()
matches["match_id"] = pd.to_numeric(matches.match_id, errors="raise").astype(int)
matches["match_date"] = pd.to_datetime(matches.match_date, errors="raise").dt.normalize()
matches[["home_score", "away_score"]] = matches[["home_score", "away_score"]].astype(int)
matches = matches.sort_values(["match_date", "kick_off", "match_id"]).reset_index(drop=True)
expected_paths = [RAW_DIR / "events" / f"{mid}.jsonl.gz" for mid in matches.match_id]
assert len(matches) == 380 and all(path.is_file() and path.stat().st_size for path in expected_paths)
display(matches.head())

## 2 · Adaptador de eventos

El adaptador admite JSON anidado del proveedor y exportaciones aplanadas de
statsbombpy. Conserva IDs del proveedor, línea y archivo fuente. Los nulos de
campos no aplicables no se imputan.

In [ ]:
def present(value):
    return value is not None and not (isinstance(value, (float, np.floating)) and np.isnan(value))


def get_name(value):
    return value.get("name") if isinstance(value, dict) else value


def get_id(value):
    return value.get("id") if isinstance(value, dict) else None


def field(raw, flat, section, key):
    value = raw.get(flat)
    return value if present(value) else (raw.get(section) or {}).get(key)


def numeric(value):
    try:
        result = float(value)
        return result if np.isfinite(result) else np.nan
    except (ValueError, TypeError):
        return np.nan


def xy(value):
    return [numeric(v) for v in value[:2]] if isinstance(value, (list, tuple)) and len(value) >= 2 else [np.nan, np.nan]


def lineup_ids(raw):
    lineup = raw.get("tactics_lineup")
    if not isinstance(lineup, list):
        lineup = (raw.get("tactics") or {}).get("lineup") or []
    result = []
    for item in lineup:
        player = item.get("player") if isinstance(item, dict) else None
        player_id = get_id(player)
        if player_id is not None:
            result.append(int(player_id))
    return result


def related_ids(raw):
    value = raw.get("related_events") or []
    return [str(item) for item in value] if isinstance(value, list) else []


def read_raw(mid):
    path = RAW_DIR / "events" / f"{mid}.jsonl.gz"
    with gzip.open(path, "rt", encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def normalize(raw, source_file, raw_line):
    location = xy(raw.get("location"))
    destination = xy(field(raw, "pass_end_location", "pass", "end_location"))
    timestamp = pd.to_timedelta(raw.get("timestamp"), errors="coerce")
    player_value = raw.get("player")
    replacement_value = field(raw, "substitution_replacement", "substitution", "replacement")
    card_value = field(raw, "foul_committed_card", "foul_committed", "card")
    if not present(card_value):
        card_value = field(raw, "bad_behaviour_card", "bad_behaviour", "card")
    team_value = raw.get("team")
    possession_team_value = raw.get("possession_team")
    type_value = raw.get("type")
    return dict(
        event_id=raw.get("id"), index=numeric(raw.get("index")),
        period=numeric(raw.get("period")), minute=numeric(raw.get("minute")),
        second=numeric(raw.get("second")),
        seconds=timestamp.total_seconds() if pd.notna(timestamp) else np.nan,
        type=get_name(type_value), type_id=get_id(type_value),
        team=get_name(team_value), team_id=get_id(team_value),
        possession_team=get_name(possession_team_value),
        possession_team_id=get_id(possession_team_value), possession=raw.get("possession"),
        player=get_name(player_value), player_id=get_id(player_value),
        recipient=get_name(field(raw, "pass_recipient", "pass", "recipient")),
        pass_type=get_name(field(raw, "pass_type", "pass", "type")),
        height=get_name(field(raw, "pass_height", "pass", "height")),
        pass_technique=get_name(field(raw, "pass_technique", "pass", "technique")),
        pass_outcome=get_name(field(raw, "pass_outcome", "pass", "outcome")),
        shot_outcome=get_name(field(raw, "shot_outcome", "shot", "outcome")),
        card=get_name(card_value), cross=field(raw, "pass_cross", "pass", "cross"),
        substitution_replacement=get_name(replacement_value),
        substitution_replacement_id=get_id(replacement_value),
        starting_xi_ids=lineup_ids(raw), related_event_ids=related_ids(raw),
        x=location[0], y=location[1], end_x=destination[0], end_y=destination[1],
        xg=numeric(field(raw, "shot_statsbomb_xg", "shot", "statsbomb_xg")),
        source_file=source_file, raw_line=raw_line,
    )


sample_id = int(matches.iloc[0].match_id)
sample = pd.DataFrame([normalize(row, f"events/{sample_id}.jsonl.gz", line)
                       for line, row in enumerate(read_raw(sample_id), 1)])
display(sample[["event_id", "index", "period", "minute", "seconds", "type", "team"]].head())
print(f"Ejemplo {sample_id}: {len(sample):,} eventos")

## 3 · Contexto inmediatamente anterior al evento

El marcador suma `Shot/Goal` y `Own Goal For`; `Own Goal Against` es el registro
recíproco y no vuelve a sumar. Los jugadores activos se reconstruyen desde
Starting XI y sustituciones. Una roja a un suplente se audita, pero no reduce
el número de jugadores en campo.

In [ ]:
DISMISSAL_CARDS = {"red card", "second yellow", "second yellow card"}


def phase_from_minute(minute):
    if minute <= 30:
        return "00-30"
    if minute <= 60:
        return "31-60"
    if minute <= 75:
        return "61-75"
    return "76+"


def build_pre_event_context(records, home_team, away_team):
    teams = [home_team, away_team]
    scores = {team: 0 for team in teams}
    active = {team: set() for team in teams}
    for event in records:
        if event["type"] == "Starting XI" and event["team"] in active:
            active[event["team"]].update(int(v) for v in event["starting_xi_ids"])
    if any(len(players) != 11 for players in active.values()):
        raise ValueError(f"Starting XI incompleto: {[(team, len(v)) for team, v in active.items()]}")

    contexts, own_goals, dismissals, substitutions = {}, [], [], []
    last_corner = {}
    for event in records:
        team = event["team"]
        opponent = next((value for value in teams if value != team), None)
        minute = float(event["minute"]) + (float(event["second"]) / 60 if np.isfinite(event["second"]) else 0)
        team_score = scores.get(team, np.nan)
        opponent_score = scores.get(opponent, np.nan)
        score_diff = team_score - opponent_score if np.isfinite(team_score) and np.isfinite(opponent_score) else np.nan
        previous = last_corner.get(team)
        delta_corner = (event["seconds"] - previous[1]) if (
            previous and previous[0] == event["period"] and np.isfinite(event["seconds"])) else np.nan
        attacking_players = len(active.get(team, set())) if team in active else np.nan
        defending_players = len(active.get(opponent, set())) if opponent in active else np.nan
        contexts[event["event_id"]] = dict(
            home_score_before=scores[home_team], away_score_before=scores[away_team],
            goals_for_before=team_score, goals_against_before=opponent_score,
            score_diff=score_diff,
            game_state="winning" if score_diff > 0 else "losing" if score_diff < 0 else "drawing",
            match_minute=minute, match_phase=phase_from_minute(minute),
            is_stoppage_time=bool((event["period"] == 1 and minute >= 45)
                                  or (event["period"] == 2 and minute >= 90)),
            attacking_players=attacking_players, defending_players=defending_players,
            player_difference=(attacking_players - defending_players)
                              if np.isfinite(attacking_players) and np.isfinite(defending_players) else np.nan,
            numerical_state="advantage" if attacking_players > defending_players else
                            "disadvantage" if attacking_players < defending_players else "equal",
            seconds_since_previous_same_team_corner=delta_corner,
            repeat_corner_60s=bool(np.isfinite(delta_corner) and 0 <= delta_corner <= 60),
            context_version=CONTEXT_VERSION,
        )

        if event["type"] == "Pass" and event["pass_type"] == "Corner":
            last_corner[team] = (event["period"], event["seconds"], event["event_id"])

        if event["type"] == "Shot" and event["shot_outcome"] == "Goal":
            scores[team] += 1
        elif event["type"] == "Own Goal For":
            scores[team] += 1
            own_goals.append({"for_event_id": event["event_id"], "beneficiary": team,
                              "period": event["period"], "seconds": event["seconds"],
                              "related_event_ids": event["related_event_ids"]})

        if event["type"] == "Substitution":
            outgoing = event["player_id"]
            replacement = event["substitution_replacement_id"]
            valid = team in active and outgoing in active[team] and replacement not in active[team]
            substitutions.append({"event_id": event["event_id"], "team": team,
                                  "outgoing_player_id": outgoing,
                                  "replacement_player_id": replacement, "valid": valid})
            if valid:
                active[team].remove(outgoing)
                active[team].add(replacement)

        card = str(event["card"] or "").strip().lower()
        if card in DISMISSAL_CARDS:
            on_pitch = team in active and event["player_id"] in active[team]
            dismissals.append({"event_id": event["event_id"], "team": team,
                               "player_id": event["player_id"], "player": event["player"],
                               "card": event["card"], "on_pitch": on_pitch})
            if on_pitch:
                active[team].remove(event["player_id"])
    return contexts, scores, own_goals, dismissals, substitutions

## 4 · Materialización y auditoría completa

Cada partido bloquea su publicación si falla identidad, orden, equipos,
transiciones de sustitución o conciliación del marcador final.

In [ ]:
audits, errors, manifest_rows = [], [], []
state_rows, own_goal_rows, dismissal_rows, substitution_rows = [], [], [], []
event_counts, seen_ids = Counter(), set()

for match in tqdm(matches.itertuples(index=False), total=len(matches), desc="Normalizar y auditar"):
    try:
        source_file = f"events/{int(match.match_id)}.jsonl.gz"
        raw = read_raw(match.match_id)
        records = [normalize(event, source_file, line) for line, event in enumerate(raw, 1)]
        records = sorted(records, key=lambda event: (event["period"], event["index"]))
        ids = [event["event_id"] for event in records]
        indices = [event["index"] for event in records]
        assert ids and all(isinstance(value, str) and value for value in ids)
        assert len(ids) == len(set(ids)) and not seen_ids.intersection(ids)
        assert len(indices) == len(set(indices))
        assert {event["team"] for event in records if event["team"]} <= {match.home_team, match.away_team}
        seen_ids.update(ids)

        context, final_scores, own_goals, dismissals, substitutions = build_pre_event_context(
            records, match.home_team, match.away_team
        )
        for event in records:
            event.update(context[event["event_id"]])
            event_counts[event["type"]] += 1
        assert final_scores[match.home_team] == int(match.home_score)
        assert final_scores[match.away_team] == int(match.away_score)
        assert all(row["valid"] for row in substitutions)

        frame = pd.DataFrame(records)
        for column in ["starting_xi_ids", "related_event_ids"]:
            frame[column] = frame[column].map(lambda value: json.dumps(value, ensure_ascii=False))
        clean_path = EVENTS_OUT / f"{int(match.match_id)}.parquet"
        frame.to_parquet(clean_path, index=False)

        regressions = sum(a["period"] == b["period"] and a["seconds"] > b["seconds"]
                          for a, b in zip(records, records[1:]))
        invalid_coords = sum(not (0 <= value <= limit) for event in records
                             for value, limit in [(event["x"], 120), (event["y"], 80),
                                                  (event["end_x"], 120), (event["end_y"], 80)]
                             if np.isfinite(value))
        corners = sum(event["type"] == "Pass" and event["pass_type"] == "Corner"
                      for event in records)
        audits.append({"match_id": int(match.match_id), "events": len(records),
                       "corners": corners, "clock_regressions": regressions,
                       "out_of_bounds_coordinates": invalid_coords})
        state_rows.append({"match_id": int(match.match_id),
                           "reconstructed_home": final_scores[match.home_team],
                           "reconstructed_away": final_scores[match.away_team],
                           "expected_home": int(match.home_score),
                           "expected_away": int(match.away_score), "score_match": True})
        own_goal_rows.extend({"match_id": int(match.match_id), **row} for row in own_goals)
        dismissal_rows.extend({"match_id": int(match.match_id), **row} for row in dismissals)
        substitution_rows.extend({"match_id": int(match.match_id), **row} for row in substitutions)
        manifest_rows.append({"match_id": int(match.match_id), "file": clean_path.name,
                              "sha256": sha256_file(clean_path), "rows": len(frame)})
    except (AssertionError, ValueError, KeyError, OSError) as exc:
        errors.append({"match_id": int(match.match_id), "error": str(exc)})

display(pd.DataFrame(errors, columns=["match_id", "error"]))
assert not errors, "La etapa 02 no publica resultados parciales"
quality = pd.DataFrame(audits)
state_quality = pd.DataFrame(state_rows)
own_goal_audit = pd.DataFrame(own_goal_rows)
dismissal_audit = pd.DataFrame(dismissal_rows)
substitution_audit = pd.DataFrame(substitution_rows)
source_manifest = pd.DataFrame(manifest_rows)
assert len(quality) == 380 and quality.events.sum() == len(seen_ids)
assert state_quality.score_match.all()
assert len(substitution_audit) == 2190 and substitution_audit.valid.all()
assert len(dismissal_audit) == 109 and dismissal_audit.on_pitch.sum() == 106
print(f"{len(quality)} partidos · {quality.events.sum():,} eventos · {quality.corners.sum():,} córners")
print(f"Autogoles: {len(own_goal_audit)} · expulsiones en campo: {dismissal_audit.on_pitch.sum()}/{len(dismissal_audit)}")
display(dismissal_audit.loc[~dismissal_audit.on_pitch])

## 5 · EDA estructural y cobertura

Los faltantes se evalúan donde el campo aplica. El volumen no se interpreta
como riqueza táctica ni eficacia.

In [ ]:
summary = pd.DataFrame({
    "regla": ["eventos", "córners", "regresiones de reloj", "coordenadas fuera de rango",
              "autogoles", "expulsiones registradas", "expulsiones en campo", "rojas fuera de campo"],
    "n": [quality.events.sum(), quality.corners.sum(), quality.clock_regressions.sum(),
          quality.out_of_bounds_coordinates.sum(), len(own_goal_audit), len(dismissal_audit),
          int(dismissal_audit.on_pitch.sum()), int((~dismissal_audit.on_pitch).sum())],
})
display(summary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
quality.events.plot.hist(bins=20, ax=axes[0], color="#246653")
axes[0].set(title="Eventos por partido", xlabel="Eventos", ylabel="Partidos")
quality.corners.plot.hist(bins=18, ax=axes[1], color="#dba43b")
axes[1].set(title="Córners detectados por partido", xlabel="Córners", ylabel="Partidos")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_volumen_estructural.png", dpi=140, bbox_inches="tight")
plt.show(); plt.close()
display(pd.Series(event_counts).sort_values(ascending=False).head(15).rename("eventos"))

## 6 · Contrato de salida

El notebook 03 debe consumir exclusivamente estas particiones. Los hashes
permiten demostrar que no se mezclaron ejecuciones.

In [ ]:
matches.to_parquet(OUT / "matches_clean.parquet", index=False)
quality.to_parquet(OUT / "data_quality.parquet", index=False)
state_quality.to_parquet(OUT / "match_state_quality.parquet", index=False)
own_goal_audit.to_parquet(OUT / "own_goal_audit.parquet", index=False)
dismissal_audit.to_parquet(OUT / "dismissal_audit.parquet", index=False)
substitution_audit.to_parquet(OUT / "substitution_audit.parquet", index=False)
source_manifest.to_csv(OUT / "source_manifest.csv", index=False)

exports = []
for name in ["matches_clean.parquet", "data_quality.parquet", "match_state_quality.parquet",
             "own_goal_audit.parquet", "dismissal_audit.parquet", "substitution_audit.parquet",
             "source_manifest.csv"]:
    path = OUT / name
    exports.append({"file": name, "sha256": sha256_file(path), "bytes": path.stat().st_size})

contract = {
    "stage": "02_clean", "contract_version": "02-clean-v2",
    "run_id": RUN_ID, "source_contract": str(ingestion_contract_path),
    "source_raw_manifest_sha256": ingestion_contract["raw_manifest_sha256"],
    "context_version": CONTEXT_VERSION,
    "counts": {"matches": 380, "events": int(quality.events.sum()),
               "corners_detected": int(quality.corners.sum()),
               "clock_regressions": int(quality.clock_regressions.sum()),
               "own_goals": int(len(own_goal_audit)),
               "substitutions": int(len(substitution_audit)),
               "dismissal_records": int(len(dismissal_audit)),
               "on_pitch_dismissals": int(dismissal_audit.on_pitch.sum()),
               "off_pitch_dismissals": int((~dismissal_audit.on_pitch).sum())},
    "event_files": 380, "event_manifest_sha256": sha256_file(OUT / "source_manifest.csv"),
    "exports": exports, "environment": ENVIRONMENT, "python": platform.python_version(),
}
(OUT / "contract.json").write_text(
    json.dumps(contract, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(pd.Series(contract["counts"]))
print("Contrato:", OUT / "contract.json")

### Conclusión

La reconstrucción coincide con los 380 marcadores oficiales. De 109 registros
de expulsión, 106 corresponden a jugadores en campo; tres rojas a suplentes no
reducen la diferencia numérica. Esta distinción corrige 23 contextos de córner
que antes se clasificaban con inferioridad inexistente.